In [2]:
import os
import nest_asyncio
from llama_index.core import SimpleDirectoryReader
from llama_index.core.node_parser import SentenceSplitter
from llama_index.llms.openai import OpenAI
from llama_index.embeddings.openai import OpenAIEmbedding
from llama_index.vector_stores.chroma import ChromaVectorStore
import chromadb
from raptor_pack.llama_index.packs.raptor.base import RaptorRetriever
from llama_index.core.query_engine import RetrieverQueryEngine
import json
from datasets import load_dataset
from llama_index.core.schema import Document
from tqdm import tqdm


from llama_index.llms.deepseek import DeepSeek

llm = DeepSeek(model="deepseek-chat", api_key=os.getenv("DS_API"))


from llama_index.embeddings.ollama import OllamaEmbedding
ollama_embedding = OllamaEmbedding(
    model_name="nomic-embed-text:latest",
    base_url="http://localhost:11434",
)

# 添加一个简单的测试
print("Testing Ollama embedding...")
try:
    test_text = "This is a test."
    embedding = ollama_embedding.get_text_embedding(test_text)
    print("Ollama embedding test successful!")
except Exception as e:
    print(f"Ollama embedding test failed: {str(e)}")


os.environ["OPENAI_API_KEY"] = ""
nest_asyncio.apply()
with open('musique_corpus.json') as file:
	data = file.read()
	lines = json.loads(data)


output_directory = 'MuSiQue_temp_data'
os.makedirs(output_directory, exist_ok=True)

all_file, count = [], 0
for value in lines:
	file_path = os.path.join(output_directory, f"{count}.txt")
	with open(file_path, 'w') as file:
		file_str = value['title'] + '\n' + value['text']
		file.write(file_str)
	all_file.append(file_path)
	count += 1

corpus = json.load(open("musique_kg.json"))
documents = []
entities_facts = {}
fact_counts = {}
for doc in corpus:
    for rel in doc["facts"]:
        documents.append(Document(text=rel["fact"]))
        for ent in rel["entities"]:
            if ent.lower() not in entities_facts:
                entities_facts[ent.lower()] = []
            entities_facts[ent.lower()] += [rel["fact"]]
            if rel["fact"] not in fact_counts:
                fact_counts[rel["fact"]] = 0
            fact_counts[rel["fact"]] += 1
new_docs = set()
for ent in entities_facts:
    if len(entities_facts[ent]) == 1 and fact_counts[entities_facts[ent][0]] > 1:
        continue
    new_docs.add("\n".join(entities_facts[ent]))
higher_level_facts = []
for doc in new_docs:
     higher_level_facts.append(Document(text=doc))

documents = SimpleDirectoryReader(input_files=all_file).load_data()
retriever = RaptorRetriever(documents, higher_level_facts=higher_level_facts, embed_model=ollama_embedding, llm=llm, similarity_top_k=20, mode="collapsed", verbose = True)
query_engine = RetrieverQueryEngine.from_args(retriever, llm=llm)

Testing Ollama embedding...
Ollama embedding test successful!
Initializing RaptorRetriever with:
- Number of input documents: 11656
- Number of higher level facts: 20788
- Tree depth: 3
- Similarity top k: 20
- Mode: collapsed
- Using LLM: DeepSeek
- Using Embedding model: OllamaEmbedding
Fact+Raptor+left_right_only_fact_aggregate!!!

=== Starting Document Processing ===
Processing 11656 documents
Processing 20788 higher level facts

=== Running Transformations ===
Transformed higher facts: 21329
Transformed documents: 11656

=== Building Fact Tree ===

Processing Level 0:
- Number of facts to process: 21329
Generating embeddings for level 0.
Processing embedding batch


100%|██████████| 427/427 [03:32<00:00,  2.01it/s]


Performing clustering for level 0.


/Users/zhangzimo/miniconda3/envs/sirerag/lib/python3.10/site-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/Users/zhangzimo/miniconda3/envs/sirerag/lib/python3.10/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/zhangzimo/miniconda3/envs/sirerag/lib/python3.10/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/zhangzimo/miniconda3/envs/sirerag/lib/python3.10/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b
/Users/zhangzimo/miniconda3/envs/sirerag/lib/python3.10/site-packages/sklearn/cluster/_kmeans.py:237: RuntimeWarning: divide by zero encountered in matmul
  current_pot = closest_dist_sq @ sample_weight
/Users/zhangzimo/miniconda3/envs/sirerag/lib/python3.10/site-packages/sklearn/

Generating summaries for level 0 with 1207 clusters.

=== 开始生成摘要 ===
总集群数: 1207
工作线程数: 16
生成摘要


...

已完成: 16/16 (100.00%)

=== 摘要生成完成 ===
成功处理: 16/16
Level 1 created summaries/clusters: 16

Processing Level 2:
- Number of nodes to process: 16
Generating embeddings for level 2.
Processing embedding batch



100%|██████████| 1/1 [00:00<00:00,  2.28it/s]
/Users/zhangzimo/miniconda3/envs/sirerag/lib/python3.10/site-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(


Performing clustering for level 2.
Generating summaries for level 2 with 4 clusters.

=== 开始生成摘要 ===
总集群数: 4
工作线程数: 16
生成摘要


100%|██████████| 1/1 [01:37<00:00, 97.11s/it]

已完成: 4/4 (100.00%)

=== 摘要生成完成 ===
成功处理: 4/4
Level 2 created summaries/clusters: 4


In [3]:
import time

with open('musique.json') as file:
	data = file.read()
	lines2 = json.loads(data)

execution_time = 0
total_eval = []
for value in lines2:
    final_question = value['question'] + " Answer this question in as fewer number of words as possible."
    start_time = time.time()
    response = query_engine.query(final_question)
    end_time = time.time()
    execution_time = execution_time + (end_time - start_time)
    element = {"q": value['question'], "a": [value['answer']] + value['answer_aliases']}
    element["predict"] = str(response).strip()
    total_eval.append(element)
    print("Finished a file!")


=== Collapsed Retrieval ===
Query: When was the person who Messi's goals in Copa del Rey compared to get signed by Barcelona? Answer this question in as fewer number of words as possible.
Retrieved 20 nodes
Finished a file!

=== Collapsed Retrieval ===
Query: What month did the Tripartite discussions begin between Britain, France, and the country where, despite being headquartered in the nation called the nobilities commonwealth, the top-ranking Warsaw Pact operatives originated? Answer this question in as fewer number of words as possible.
Retrieved 20 nodes
Finished a file!

=== Collapsed Retrieval ===
Query: What county is Erik Hort's birthplace a part of? Answer this question in as fewer number of words as possible.
Retrieved 20 nodes
Finished a file!

=== Collapsed Retrieval ===
Query: What year did the publisher of Labyrinth end? Answer this question in as fewer number of words as possible.
Retrieved 20 nodes
Finished a file!

=== Collapsed Retrieval ===
Query: When was Lady God

BadRequestError: Error code: 400 - {'error': {'message': 'Content Exists Risk', 'type': 'invalid_request_error', 'param': None, 'code': 'invalid_request_error'}}

In [4]:
with open('output/output_musique_SiReRAG_gpt4o_temp0.json', 'w') as file:
    file.write(json.dumps(total_eval))